# Calculate effective coverage by quintile and scenario

Also by age, sex, and pregnancy status (though coverage will not vary by pregnancy status due to a lack of data).

This is similar to what the pregnancy simulation does at the individual level, but using groups instead.
It can be shared between multiplication models that do not incorporate individual heterogeneity.

In [1]:
import pandas as pd

In [2]:
location = "nigeria"
vehicle = "rice"
fortificant = "iron"

In [3]:
# Parameters
location = "india"
fortificant = "folate"
vehicle = "wheat"


In [4]:
results_dir = f"../results/{fortificant}/{vehicle}"

In [5]:
full_coverage_probability = pd.read_csv(
    f"{results_dir}/baseline_fortification/full_coverage/{location}.csv"
)
full_coverage_probability = full_coverage_probability.set_index(
    [c for c in full_coverage_probability.columns if c != "value"]
).value
full_coverage_probability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                wheat           0.026679
                            2                wheat           0.025336
                            3                wheat           0.025627
                            4                wheat           0.027411
                            5                wheat           0.029122
        5          15       1                wheat           0.026142
                            2                wheat           0.025209
                            3                wheat           0.026487
                            4                wheat           0.027896
                            5                wheat           0.029402
        15         30       1                wheat           0.025832
                            2                wheat           0.024928
                            3                wheat           0.026191
                            4   

In [6]:
any_coverage_probability = pd.read_csv(
    f"{results_dir}/baseline_fortification/any_coverage/{location}.csv"
)
any_coverage_probability = any_coverage_probability.set_index(
    [c for c in any_coverage_probability.columns if c != "value"]
).value
any_coverage_probability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                wheat           0.026679
                            2                wheat           0.025336
                            3                wheat           0.025627
                            4                wheat           0.027411
                            5                wheat           0.029122
        5          15       1                wheat           0.026142
                            2                wheat           0.025209
                            3                wheat           0.026487
                            4                wheat           0.027896
                            5                wheat           0.029402
        15         30       1                wheat           0.025832
                            2                wheat           0.024928
                            3                wheat           0.026191
                            4   

In [7]:
partial_coverage_mean = pd.read_csv(
    f"{results_dir}/baseline_fortification/partial_coverage_amount/mean/{location}.csv"
)
partial_coverage_mean = partial_coverage_mean.set_index(
    [c for c in partial_coverage_mean.columns if c != "value"]
).value
partial_coverage_mean

wealth_quintile  vehicle_name
1                wheat           0
2                wheat           0
3                wheat           0
4                wheat           0
5                wheat           0
Name: value, dtype: int64

In [8]:
current_coverage = (
    full_coverage_probability
    + (any_coverage_probability - full_coverage_probability) * partial_coverage_mean
)
current_coverage

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                wheat           0.026679
                            2                wheat           0.025336
                            3                wheat           0.025627
                            4                wheat           0.027411
                            5                wheat           0.029122
        5          15       1                wheat           0.026142
                            2                wheat           0.025209
                            3                wheat           0.026487
                            4                wheat           0.027896
                            5                wheat           0.029402
        15         30       1                wheat           0.025832
                            2                wheat           0.024928
                            3                wheat           0.026191
                            4   

In [9]:
from lsff_utils import config_utils

scenarios = config_utils.get_intervention_scenarios(location, vehicle)
scenarios

['intervention']

In [10]:
any_consumption = pd.read_csv(
    f"../results/{vehicle}/vehicle_consumption/any/{location}.csv"
)
any_consumption = any_consumption.set_index(
    [c for c in any_consumption.columns if c != "value"]
).value
any_consumption

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                wheat           0.814343
                            2                wheat           0.842671
                            3                wheat           0.836465
                            4                wheat           0.828110
                            5                wheat           0.838037
        5          15       1                wheat           0.890750
                            2                wheat           0.951041
                            3                wheat           0.938737
                            4                wheat           0.915622
                            5                wheat           0.960267
        15         30       1                wheat           0.889706
                            2                wheat           0.931708
                            3                wheat           0.937345
                            4   

In [11]:
if (
    len(
        any_consumption.reset_index()[["sex", "age_start", "age_end"]].drop_duplicates()
    )
    == 1
):
    # Assumed same for all ages/sexes
    any_consumption = any_consumption.droplevel(["sex", "age_start", "age_end"])
    display(any_consumption)

In [12]:
fortifiability = pd.read_csv(
    f"../results/{vehicle}/vehicle_consumption/fortifiability/{location}.csv"
)
fortifiability = fortifiability.set_index(
    [c for c in fortifiability.columns if c != "value"]
).value
fortifiability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                wheat           0.119459
                            2                wheat           0.113446
                            3                wheat           0.114749
                            4                wheat           0.122737
                            5                wheat           0.130395
        5          15       1                wheat           0.117055
                            2                wheat           0.112876
                            3                wheat           0.118597
                            4                wheat           0.124908
                            5                wheat           0.131649
        15         30       1                wheat           0.115668
                            2                wheat           0.111617
                            3                wheat           0.117275
                            4   

In [13]:
import pathlib, numpy as np

for scenario in scenarios:
    intervention_coverage = pd.read_csv(
        f"{results_dir}/{scenario}/intervention_fortification/any_coverage/{location}.csv"
    )
    intervention_coverage = intervention_coverage.set_index(
        [c for c in intervention_coverage.columns if c != "value"]
    ).value
    target_coverage = intervention_coverage * fortifiability
    display(target_coverage)
    assert (
        (target_coverage > current_coverage.reindex_like(target_coverage))
        | np.isclose(target_coverage, current_coverage.reindex_like(target_coverage))
    ).all()
    target_coverage[
        np.isclose(target_coverage, current_coverage.reindex_like(target_coverage))
    ] = current_coverage.reindex_like(target_coverage)
    # Not all coverage is effective -- this is as a proportion of coverage!
    effectiveness = pd.read_csv(
        f"{results_dir}/{scenario}/intervention_fortification/effectiveness/{location}.csv"
    )
    effectiveness = effectiveness.set_index(
        [c for c in effectiveness.columns if c != "value"]
    ).value
    effective_intervention_coverage = any_consumption * target_coverage * effectiveness
    path = f"{results_dir}/{scenario}/intervention_fortification/effective_coverage/{location}.csv"
    pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
    effective_intervention_coverage.reset_index().to_csv(path, index=False)

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                wheat           0.109903
                            2                wheat           0.104370
                            3                wheat           0.105569
                            4                wheat           0.112918
                            5                wheat           0.119963
        5          15       1                wheat           0.107691
                            2                wheat           0.103846
                            3                wheat           0.109109
                            4                wheat           0.114916
                            5                wheat           0.121117
        15         30       1                wheat           0.106414
                            2                wheat           0.102688
                            3                wheat           0.107893
                            4   

In [14]:
# Not all coverage is effective -- this is as a proportion of coverage!
effectiveness = pd.read_csv(
    f"{results_dir}/baseline_fortification/effectiveness/{location}.csv"
)
effectiveness = effectiveness.set_index(
    [c for c in effectiveness.columns if c != "value"]
).value

In [15]:
effective_baseline_coverage = any_consumption * current_coverage * effectiveness
effective_baseline_coverage

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                wheat           0.004780
        5          15       1                wheat           0.005123
        15         30       1                wheat           0.005056
        30         50       1                wheat           0.005184
        50         125      1                wheat           0.004900
Male    0          5        1                wheat           0.004730
        5          15       1                wheat           0.005206
        15         30       1                wheat           0.005004
        30         50       1                wheat           0.005184
        50         125      1                wheat           0.004897
Female  0          5        2                wheat           0.004697
        5          15       2                wheat           0.005274
        15         30       2                wheat           0.005110
        30         50       2   

In [16]:
path = f"{results_dir}/baseline_fortification/effective_coverage/{location}.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
effective_baseline_coverage.reset_index().to_csv(path, index=False)